In [ ]:
import requests
import pandas as pd
import io
from dotenv import load_dotenv
import os

load_dotenv();
api_key= os.getenv("loc7")

# 1. Construct the Batch URL
url = api_key

# 2. Coordinates extracted from your link (7 Locations)
lats = [25.1825, 23.1670, 25.5941, 22.5626, 26.1844, 23.8361, 28.4190]
lons = [75.8391, 79.9501, 85.1356, 88.3630, 91.7458, 91.2794, 95.0893]

# Convert to comma-separated strings
lat_str = ",".join(map(str, lats))
lon_str = ",".join(map(str, lons))

# 3. CLEANED Variables
# Removed 'winddirection_10m_dominant' (typo) and 'hourly' (unused)
daily_vars = (
    "weather_code,temperature_2m_max,temperature_2m_min,apparent_temperature_max,"
    "apparent_temperature_min,sunshine_duration,daylight_duration,sunrise,sunset,"
    "precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,shortwave_radiation_sum,"
    "wind_direction_10m_dominant,wind_speed_10m_max,et0_fao_evapotranspiration,"
    "cloud_cover_max,cloud_cover_min,temperature_2m_mean,cloud_cover_mean,"
    "soil_moisture_0_to_100cm_mean,dew_point_2m_mean,dew_point_2m_max,dew_point_2m_min,"
    "surface_pressure_min,surface_pressure_max,pressure_msl_min,pressure_msl_max,"
    "pressure_msl_mean,relative_humidity_2m_min,relative_humidity_2m_max,"
    "relative_humidity_2m_mean,soil_temperature_0_to_100cm_mean,wind_speed_10m_min,"
    "wind_speed_10m_mean,wet_bulb_temperature_2m_min,wet_bulb_temperature_2m_max,"
    "wet_bulb_temperature_2m_mean"
)



print(f"Fetching data for {len(lats)} locations...")

response = requests.get(url)

if response.status_code != 200:
    print(f"Error! Status Code: {response.status_code}")
    print("Reason:", response.text)
else:
    data = response.json()
    
    # Handle Batch Response (List of results)
    if isinstance(data, list):
        results_list = data
    else:
        results_list = [data]

    all_dfs = []

    for i, location_data in enumerate(results_list):
        if "daily" not in location_data:
            print(f"Skipping location {i}: No daily data found.")
            continue
            
        df = pd.DataFrame(location_data["daily"])
        
        # Add metadata
        df["latitude"] = location_data.get("latitude")
        df["longitude"] = location_data.get("longitude")
        df["location_id"] = i + 1 
        
        # Convert time to datetime
        df['time'] = pd.to_datetime(df['time'])
        
        all_dfs.append(df)

    # Combine and Save
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        
        # Set display options to show all columns
        pd.set_option('display.max_columns', None)
        
        print("Success!")
        print(f"Total Shape: {final_df.shape}")
        print(final_df.head())
        
        # Save to CSV
        filename = 'weather_data_7_locations.csv'
        final_df.to_csv(filename, index=False)
        print(f"Saved to '{filename}'")
    else:
        print("No data frames were created.")

Fetching data for 7 locations...
Success!
Total Shape: (17899, 43)
        time  weather_code  temperature_2m_max  temperature_2m_min  \
0 2018-01-01             0                24.4                11.4   
1 2018-01-02             0                22.8                 8.6   
2 2018-01-03             0                22.9                10.3   
3 2018-01-04             0                24.4                 8.1   
4 2018-01-05             0                24.0                 9.9   

   apparent_temperature_max  apparent_temperature_min  sunshine_duration  \
0                      23.5                      10.1           35208.99   
1                      20.5                       7.1           35233.59   
2                      20.4                       8.2           35260.27   
3                      22.0                       5.7           35289.14   
4                      22.4                       8.7           35320.67   

   daylight_duration           sunrise            sunse

In [1]:
!pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 6


https://archive-api.open-meteo.com/v1/archive?latitude={lat_str}&longitude={lon_str}&start_date=2018-01-01&end_date=2024-12-31&daily={daily_vars}&timezone=auto
